# Semi-Supervised Classification with Graph Convolutional Networks (GCN)
## Replication & Validation Study of Kipf & Welling (ICLR 2017)
**Author:** Eugene Phang  
**Original Paper:** [Semi-Supervised Classification with Graph Convolutional Networks (arXiv:1609.02907)](https://arxiv.org/abs/1609.02907)  
**Repository Reference:** [tkipf/gcn (GitHub)](https://github.com/tkipf/gcn)

---

### Executive Summary
This notebook provides an independent PyTorch & PyTorch Geometric replication of the seminal 2017 paper by Thomas Kipf and Max Welling. The goal is to benchmark a multi-layer Spectral Graph Convolutional Network (GCN) on semi-supervised node classification across three canonical benchmark citation networks: **Cora**, **Citeseer**, and **Pubmed**.


In [27]:
# Setup & Package Installation
# PyTorch Geometric provides optimized GPU kernels for message passing and graph convolutions
!pip install -q torch-geometric scikit-learn matplotlib


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
# Check PyTorch and GPU acceleration
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")


In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np

In [30]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


cuda


The Planetoid constructor downloads the same dataset formatting found in:
https://github.com/tkipf/gcn/tree/master/gcn/data

Details of the matrices and test-validation-training partitionings can be found here:

https://github.com/tkipf/gcn/blob/master/gcn/utils.py

Note: the adjacency matrix construction is self implied by the dataset preparation seeing as the edge index is supplied directly by Planetoid.

Details on building an adjacency matrix for use by the GCN codebase are provided but this paper concerns itself with validating the PyTorch implementation against the reported performance of the original paper.


In [31]:
# Load datasets
dataset_cora = Planetoid(root='/tmp/Cora', name='Cora')
dataset_pubmed = Planetoid(root='/tmp/Pubmed', name='Pubmed')
dataset_citeseer = Planetoid(root='/tmp/Citeseer', name='Citeseer')

datasets = {'Citeseer': dataset_citeseer, 'Cora': dataset_cora, 'Pubmed': dataset_pubmed}

The GCN neural net is made of two convolution layers interspersed with a RELU and then a Softmax final output.
In: https://github.com/tkipf/gcn/blob/master/gcn/models.py are details of the layers in the class definition of GCN.

I've added here a peek_embeddings flag to examine the subfeatures produced by the model which will be useful for building architectures to examine edge-prediction tasks instead of node-classification.

```
#excerpt from Kipf and Welling 2017
    def _build(self):

        self.layers.append(GraphConvolution(input_dim=self.input_dim,
                                            output_dim=FLAGS.hidden1,
                                            placeholders=self.placeholders,
                                            act=tf.nn.relu,
                                            dropout=True,
                                            sparse_inputs=True,
                                            logging=self.logging))

        self.layers.append(GraphConvolution(input_dim=FLAGS.hidden1,
                                            output_dim=self.output_dim,
                                            placeholders=self.placeholders,
                                            act=lambda x: x,
                                            dropout=True,
                                            logging=self.logging))

    def predict(self):
        return tf.nn.softmax(self.outputs)
```



In [35]:
# Define GCN model
class GCN(torch.nn.Module):
    def __init__(self, num_features, hidden_dim, num_classes):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)

    def forward(self, x, edge_index, peek_embeddings=False):
        x = F.relu(self.conv1(x, edge_index))
        if peek_embeddings:
            print(x.shape)  #16-dimensional node embeddings
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

Hyperparameters were gleaned from:
https://github.com/tkipf/gcn/blob/master/gcn/train.py
The "optimizer" backpropagates learning rate and decay lossage as defined in the Tensorflow flags.

See the "forward" PyTorch method for dropout values matching with these flags.

The feature size of 16 specified as "hidden1" is passed to the PyTorch GCN constructor as "hidden_dim"
and is instantiated twice with the same value of 16.

I've added F1 Macro averaging (specified as 'weighted' by sklearns utility function) in lieu of the unbalanced class sizes of each dataset and also to clarify against the Training Accuracy readings of 1.0 for each of the training runs.

Each loss calculation per epoch matches the performance curves graphed in Kipf and Welling.
```

flags.DEFINE_float('learning_rate', 0.01, 'Initial learning rate.')
flags.DEFINE_integer('epochs', 200, 'Number of epochs to train.')
flags.DEFINE_integer('hidden1', 16, 'Number of units in hidden layer 1.')
flags.DEFINE_float('dropout', 0.5, 'Dropout rate (1 - keep probability).')
flags.DEFINE_float('weight_decay', 5e-4, 'Weight for L2 loss on embedding matrix.')
flags.DEFINE_integer('early_stopping', 10, 'Tolerance for early stopping (# of epochs).')
flags.DEFINE_integer('max_degree', 3, 'Maximum Chebyshev polynomial degree.')

```




In [36]:
# Training and Evaluation loop for each dataset
results = {}  # Dictionary to store results
for name, dataset in datasets.items():
    print(f"--- Training and Evaluating on {name} dataset ---")
    data = dataset[0].to(device)

    # Initialize model, optimizer for each dataset
    model = GCN(num_features=dataset.num_features,
                hidden_dim=16,
                num_classes=dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    # Training loop
    model.train()
    for epoch in range(200):
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 20 == 0: # Print loss every 20 epochs
            print(f'Epoch {epoch+1}, Loss: {loss:.4f}' )

    # Evaluation
    model.eval()
    pred = model(data.x, data.edge_index).argmax(dim=1)
    y_true = data.y.cpu().numpy()
    y_pred = pred.cpu().numpy()

    train_acc = accuracy_score(y_true[data.train_mask.cpu()], y_pred[data.train_mask.cpu()])
    val_acc = accuracy_score(y_true[data.val_mask.cpu()], y_pred[data.val_mask.cpu()])
    test_acc = accuracy_score(y_true[data.test_mask.cpu()], y_pred[data.test_mask.cpu()])

    test_f1 = f1_score(y_true[data.test_mask.cpu()], y_pred[data.test_mask.cpu()], average='weighted') #unbalanced class distribution

    probs = model(data.x, data.edge_index).exp()  # Convert log_softmax back to probabilities
    y_probs = probs[data.test_mask].cpu().detach().numpy()
    y_true_test = y_true[data.test_mask.cpu()]

    test_auc = roc_auc_score(y_true_test, y_probs, multi_class='ovr', average='macro')

    results[name] = {
        'Train Accuracy': train_acc,
        'Validation Accuracy': val_acc,
        'Test Accuracy': test_acc,
        'F1 Macro': test_f1,
        'ROC AUC': test_auc
    }


    print(f"\nResults for {name}:")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f">>> Test Accuracy: {test_acc:.4f}")
    print(f"F1 Macro: {test_f1:.4f}")
    print(f"ROC AUC: {test_auc:.4f}")

    # Overfitting diagnostic
    if train_acc - val_acc > 0.1:
        print("High variance.Possible overfit? Check sample sizes.")
    else:
        print("Balanced")
    print("-" * 50)

--- Training and Evaluating on Citeseer dataset ---
Epoch 20, Loss: 0.1177
Epoch 40, Loss: 0.0583
Epoch 60, Loss: 0.0348
Epoch 80, Loss: 0.0451
Epoch 100, Loss: 0.0403
Epoch 120, Loss: 0.0242
Epoch 140, Loss: 0.0267
Epoch 160, Loss: 0.0243
Epoch 180, Loss: 0.0329
Epoch 200, Loss: 0.0249

Results for Citeseer:
Train Accuracy: 1.0000
Validation Accuracy: 0.6680
>>> Test Accuracy: 0.6810
F1 Macro: 0.6896
ROC AUC: 0.8785
High variance.Possible overfit? Check sample sizes.
--------------------------------------------------
--- Training and Evaluating on Cora dataset ---
Epoch 20, Loss: 0.3414
Epoch 40, Loss: 0.0703
Epoch 60, Loss: 0.0757
Epoch 80, Loss: 0.0518
Epoch 100, Loss: 0.0411
Epoch 120, Loss: 0.0311
Epoch 140, Loss: 0.0357
Epoch 160, Loss: 0.0295
Epoch 180, Loss: 0.0368
Epoch 200, Loss: 0.0495

Results for Cora:
Train Accuracy: 1.0000
Validation Accuracy: 0.7780
>>> Test Accuracy: 0.8030
F1 Macro: 0.8041
ROC AUC: 0.9633
High variance.Possible overfit? Check sample sizes.
-----------

In [37]:
print("\n--- Final Test Accuracy Report ---")
expected_accuracies = {'Citeseer': 0.703, 'Cora': 0.815, 'Pubmed': 0.790}
for name, metrics in results.items():
    calculated_accuracy = metrics['Test Accuracy']
    expected_accuracy = expected_accuracies.get(name, None)
    if expected_accuracy is not None:
        print(f"{name:<10}: Calculated Test Accuracy = {calculated_accuracy:.3f}, Expected Test Accuracy = {expected_accuracy:.3f}")
    else:
        print(f"{name:<10}: Calculated Test Accuracy = {calculated_accuracy:.2f}")


--- Final Test Accuracy Report ---
Citeseer  : Calculated Test Accuracy = 0.681, Expected Test Accuracy = 0.703
Cora      : Calculated Test Accuracy = 0.803, Expected Test Accuracy = 0.815
Pubmed    : Calculated Test Accuracy = 0.790, Expected Test Accuracy = 0.790


The comparison outputs hover around somewhat but the proportions of accuracy readings do indeed reflect the same patterns produced by Kipf and Welling.